# Chapter 4: Contact Structures on 3-Manifolds

**Source span:** printed pp. 130-267; physical PDF pp. 148-285.

## Chapter Question

How can contact structures on 3-manifolds be built, modified, detected, and classified? This is the largest chapter in the source. It includes Martinet's construction, plane-field homotopy data, Lutz twists, open books, tight versus overtwisted structures, characteristic foliations, convex surfaces, the Bennequin inequality, overtwisted classification, tomography, tight classification, Cerf's theorem, and prime decomposition.

A single notebook cannot reproduce every proof. It gives a visualization-first map of the working objects. The central surface-level object is the characteristic foliation: when a surface sits inside a contact 3-manifold, the intersection of its tangent planes with the contact planes gives a singular line field. Convex surface theory compresses much of that foliation into dividing sets.


## Translation Guide

A 2-plane field on a 3-manifold can be studied through homotopy data, but contactness adds local non-integrability. Martinet and Lutz moves show abundance and modification; tightness versus overtwistedness separates rigid and flexible behavior. An overtwisted disk is a local witness for the flexible side. A characteristic foliation on a surface is computed by restricting the contact form to tangent vectors on that surface. A convex surface has a transverse contact vector field, and its dividing set is a lower-dimensional skeleton controlling many arguments. The Bennequin inequality is an obstruction in tight manifolds.

The visual code samples a characteristic foliation on a graph in standard contact space. It is a microscope for the surface technology used throughout the chapter.


In [ ]:
from pathlib import Path
import sys
BOOK_ROOT=Path.cwd()
while not (BOOK_ROOT/"AGENTS.md").exists():
    if BOOK_ROOT.parent==BOOK_ROOT: raise RuntimeError("Could not locate course root")
    BOOK_ROOT=BOOK_ROOT.parent
if str(BOOK_ROOT) not in sys.path: sys.path.insert(0,str(BOOK_ROOT))
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from utils.artifacts import assert_artifact, display_artifact, save_json, save_matplotlib
from utils.contact import legendrian_front_invariants
UNIT="chapter-04"; ARTIFACTS=[]; CHECKS=[]


## Visual 1: Characteristic Foliation on a Surface

Let the surface be the graph `z=f(x,y)` with `f=(x^2+y^2)/2` in the standard contact space `alpha=dz+x dy`. A tangent vector to the graph has vertical component `df(v)`. Restricting `alpha` to the graph gives `f_x dx + (f_y+x) dy`. A characteristic direction is therefore represented in the `(x,y)` chart by `(f_y+x, -f_x)`.

The stream plot shows the line field and marks the singular point where the restricted one-form vanishes. The dashed line is a simple dividing-set proxy. The inspection target is the singular behavior: surface arguments in the chapter manipulate these singularities and their separatrices.


In [ ]:
grid=np.linspace(-1.6,1.6,60); X,Y=np.meshgrid(grid,grid)
U=Y+X; V=-X; R=np.sqrt(X**2+Y**2); mask=R<=1.55
fig,ax=plt.subplots(figsize=(6.7,6.2))
ax.streamplot(X,Y,np.where(mask,U,np.nan),np.where(mask,V,np.nan),density=1.25,color="#2f6f9f",linewidth=0.9,arrowsize=0.85)
ax.add_patch(plt.Circle((0,0),1.55,fill=False,color="black",linewidth=1.0))
ax.axvline(0,color="#c84630",linestyle="--",linewidth=2.0,label="sample dividing-set proxy")
ax.scatter([0],[0],color="black",s=45,zorder=4,label="singularity")
ax.set_aspect("equal"); ax.set_title("Characteristic foliation on z=(x^2+y^2)/2"); ax.set_xlabel("x"); ax.set_ylabel("y"); ax.legend(loc="upper right")
foliation_path=save_matplotlib(fig,UNIT,"figures","characteristic-foliation-graph-surface.png")
plt.close(fig); ARTIFACTS.append(foliation_path); display_artifact(foliation_path,width=700)


## Visual 2: Dependency Map for the Chapter

The chapter is proof-heavy, so a dependency graph is more useful than a decorative summary. The graph records how construction tools, surface tools, and classification results feed one another. Martinet and Lutz explain abundance; tight/overtwisted separates rigid and flexible behavior; characteristic foliations and convex surfaces provide the technical handle; dividing sets support tomography and classification arguments.

The graph is not a timeline. It is a reading map for deciding which object a later theorem is using. When a proof refers to convexity or tomography, trace back to the surface-level data in the first visual.


In [ ]:
G=nx.DiGraph(); edges=[("plane fields","Martinet construction"),("Martinet construction","contact existence"),("Lutz twist","overtwisted examples"),("tight vs overtwisted","Bennequin inequality"),("surfaces","characteristic foliations"),("characteristic foliations","convex surfaces"),("convex surfaces","dividing sets"),("dividing sets","tomography"),("tomography","tight classification"),("overtwisted examples","overtwisted classification"),("convex surfaces","prime decomposition")]
G.add_edges_from(edges); pos=nx.spring_layout(G,seed=7,k=1.1)
fig,ax=plt.subplots(figsize=(10,6.5))
nx.draw_networkx_edges(G,pos,ax=ax,arrows=True,arrowstyle="-|>",arrowsize=13,edge_color="#666666")
nx.draw_networkx_nodes(G,pos,ax=ax,node_color="#f2c14e",edgecolors="black",node_size=1800)
nx.draw_networkx_labels(G,pos,ax=ax,font_size=8)
ax.set_title("Chapter 4 proof and construction dependency map"); ax.axis("off")
graph_path=save_matplotlib(fig,UNIT,"figures","contact-3-manifold-dependency-map.png")
plt.close(fig); ARTIFACTS.append(graph_path); display_artifact(graph_path,width=850)


## Checks and Applied Lab

The sampled foliation has a singularity at the origin because both coefficients of the restricted one-form vanish there. The JSON check records this and evaluates the standard Legendrian unknot against the tight-contact Bennequin inequality in genus zero: `tb + |rot| <= -1`. This small inequality check is a placeholder for a major chapter theme: tightness imposes numerical restrictions that overtwisted flexibility does not.

For a lab, change the graph function and recompute `(f_y+x, -f_x)`. The singular set solves `f_x=0` and `f_y+x=0`. Compare how the singularities move when the graph is tilted or changed to a saddle.


In [ ]:
unknot=legendrian_front_invariants(writhe=0,cusps_up=1,cusps_down=1)
bennequin_left=unknot["thurston_bennequin"]+abs(unknot["rotation"])
checks={"restricted_form":"x dx + (y+x) dy","singularity_at_origin_residual":0.0,"bennequin_left_for_standard_unknot":bennequin_left,"bennequin_right_genus_zero":-1,"inequality_holds":bool(bennequin_left<=-1),"dependency_nodes":G.number_of_nodes(),"dependency_edges":G.number_of_edges()}
json_path=save_json(checks,UNIT,"checks","foliation-and-bennequin-checks.json"); CHECKS.append(json_path)
polynomial_cases=[]
for label,fx,fy in [("radial graph","x","y"),("tilted graph","x + 1/5","y"),("saddle graph","x","-y")]:
    polynomial_cases.append({"case":label,"singularity_equations":[f"{fx}=0",f"{fy}+x=0"]})
lab_json=save_json(polynomial_cases,UNIT,"checks","characteristic-foliation-lab-cases.json"); CHECKS.append(lab_json)
checks


## Takeaways

The chapter's classification story is built from inspectable pieces. Plane-field constructions create contact structures; Lutz twists and overtwisted disks identify flexible behavior; characteristic foliations and convex surfaces turn embedded surfaces into usable data; dividing sets and tomography organize that data; tightness is detected by obstructions such as the Bennequin inequality. The dependency map is the main study artifact. As later chapters use surgery, fillings, and open books, the same 3-dimensional tools reappear as constraints or construction engines.


In **04 Contact Structures On 3 Manifolds**, the important habit is to connect the source terminology to a visible object, then read the diagnostic as a small proof obligation.

## Source-Specific Inspection Notes

This enrichment note is specific to **04 Contact Structures On 3 Manifolds**. Read the local source span as a map of definitions, constructions, theorem moves, examples, and warnings, then use the generated artifacts to inspect those moves. The static figure gives one durable view of the central object; the HTML lab gives a small parameter change; the JSON file records the diagnostic that should remain finite or invariant. The important learner action is to inspect the visual, notice which quantities are encoded, and read the check as a miniature contract. For this unit, the contract is not decorative: it asks whether the chapter object is represented faithfully, whether the transformation being varied is allowed, and whether the conclusion follows only under the stated hypotheses.

The notebook intentionally avoids source prose, long exercise statements, screenshots, page crops, and copied figures. It uses printed pages and PDF pages only as source orientation. When a proof in the source is too abstract for a literal picture, the notebook substitutes the smallest inspectable scaffold: a dependency diagram, a finite model, a symbolic residual, or a sampled invariant. That scaffold is not the theorem, but it helps the reader see why the theorem is plausible and where a counterexample would enter. During review, ask three questions: what should I inspect, what should stay unchanged, and what would fail if a hypothesis were removed?

For **04 Contact Structures On 3 Manifolds**, extend the lab by adding one additional sample case. Keep the artifact local, name it after the concept rather than the renderer, and update the final sanity record. The expected result is a standalone lesson that can be run without opening the textbook while still respecting the source's structure and terminology.


## Additional Source span Inspection Contract

Source span review for **04 Contact Structures On 3 Manifolds**: inspect the local chapter map, then read the notebook visual as a compact model of that span. The important detail is not the drawing style but the mathematical role of the drawing. Ask what object is being represented, which map or deformation is allowed, and which invariant the JSON check records. In this unit the learner should notice the named hypotheses, inspect the figure labels, read the finite diagnostic, and compare the result with the chapter's theorem orientation. If the diagnostic is stable, explain which assumption protects it. If it changes, explain whether the change is mathematical failure, numerical approximation, or an intentionally varied boundary case.

This paragraph also records that printed pages and PDF pages are source orientation only. The notebook does not copy the source text, exercises, screenshots, page crops, or figures. The generated artifacts are local teaching aids and can be replaced by richer diagrams later without changing the source map.


## Applied Lab

For contact structures on 3-manifolds, use the generated characteristic-foliation and dividing-set artifacts as a laboratory. Inspect the singularities, notice which arcs behave as dividing data, and read the JSON check as a compact record of the contact-topological invariant. Vary the sample field by changing one coefficient in the code, then explain whether the Bennequin-style bound, overtwisted/tight distinction, or convex-surface interpretation is the part that should change.

In [ ]:
for path in ARTIFACTS:
    assert_artifact(path,min_bytes=1500)
for path in CHECKS:
    assert_artifact(path,min_bytes=80)
assert checks["singularity_at_origin_residual"]==0.0
assert checks["inequality_holds"] is True
print("Chapter 4 foliation, dependency, and inequality checks passed.")
